# LLM vs GAN: Agent4SR vs AUSH Profile Analysis

**Research question**: *Why are LLM-generated shilling profiles harder for existing detectors to catch?*

This notebook compares three groups on four axes:

| Group | Attack | Colour |
|-------|--------|--------|
| Real users | — | Gray |
| Agent4SR | LLM-agent (GPT-4o / GPT-2 proxy) | Purple |
| AUSH | GAN (rating vectors only) | Coral |

**Figures produced**:
1. `llm_vs_gan_perplexity.pdf` — GPT-2 perplexity boxplot
2. `llm_vs_gan_ngram.pdf` — Type-token ratio & distinct-2
3. `llm_vs_gan_rating_kl.pdf` — KL divergence from real rating distribution
4. `llm_vs_gan_tsne.pdf` — t-SNE of BERT CLS embeddings (500 samples each)

> **Note on Agent4SR proxy**: If `OPENAI_API_KEY` is unset, review texts are generated
> using GPT-2 conditioned on rating-appropriate prompts.  GPT-2–generated text and
> GPT-4o–generated text share the key statistical property (low perplexity under any
> LM, diverse n-grams) that makes them hard to distinguish from real user reviews.

In [ ]:
# ── stdlib ────────────────────────────────────────────────────────────────
import os, sys, warnings, random
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..').resolve()))
os.chdir(Path('..').resolve())   # make relative imports predictable

# ── numerics ──────────────────────────────────────────────────────────────
import numpy as np
import torch
from scipy.stats import entropy as scipy_entropy
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# ── HuggingFace ───────────────────────────────────────────────────────────
from transformers import (
    GPT2LMHeadModel, GPT2TokenizerFast,
    AutoModel, AutoTokenizer,
    set_seed as hf_set_seed,
)

# ── plotting ──────────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker

matplotlib.rcParams.update({
    'font.family':       'serif',
    'font.size':         11,
    'axes.titlesize':    11,
    'axes.labelsize':    10,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'legend.fontsize':   9,
    'pdf.fonttype':      42,   # embed TrueType in PDF
    'ps.fonttype':       42,
})

# ── project imports ───────────────────────────────────────────────────────
from src.attacks import AUSHAttack, RandomAttack
from src.attacks.base import FakeProfile
from src.datasets import (
    AmazonBooksLoader, MovieLens1MLoader,
    AmazonClothingLoader, Yelp2018Loader,
)

# ── constants ─────────────────────────────────────────────────────────────
SEED          = 42
N_PROFILES    = 150      # fake profiles per attack
BUDGET_PCT    = 0.05     # filler fraction
N_TARGETS     = 5        # target items
N_SAMPLE      = 500      # samples per group for t-SNE
AUSH_EPOCHS   = 10       # fast GAN pretrain
AUSH_MAX_U    = 500      # cap training users

COLOR_REAL     = '#888888'
COLOR_AGENT4SR = '#7B2D8B'
COLOR_AUSH     = '#E8604C'

FIGURES_DIR = Path('results/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(f'Device: {DEVICE}')
print(f'Figures → {FIGURES_DIR.resolve()}')

## 1 · Dataset Loading

Prefer Amazon-Books (rich review text corpus); fall back to ML-1M → Amazon-Clothing
→ synthetic data if none are present.

In [ ]:
_LOADERS = [
    ('amazon_books',    AmazonBooksLoader),
    ('amazon_clothing', AmazonClothingLoader),
    ('ml1m',            MovieLens1MLoader),
    ('yelp2018',        Yelp2018Loader),
]

dataset = None
for name, Loader in _LOADERS:
    try:
        dataset = Loader(raw_dir='data/raw').load()
        print(f'Loaded {name}: {dataset.num_users:,} users · '
              f'{dataset.num_items:,} items · {dataset.num_reviews:,} reviews')
        has_real_text = any(t.strip() for t in dataset.review_texts[:100])
        print(f'  Has review text: {has_real_text}')
        break
    except Exception as exc:
        print(f'  {name} unavailable: {exc}')

if dataset is None:
    print('\nNo real dataset found — building synthetic corpus.')

    # ── Synthetic fallback ──────────────────────────────────────────────
    # 50 diverse review sentences spanning 1-5 stars
    _SYNTH_REVIEWS = [
        'A truly exceptional read that I could not put down.',
        'The narrative arc is compelling and the characters well developed.',
        'Beautifully written with rich, evocative prose throughout.',
        'I was completely absorbed from the opening chapter to the final page.',
        'One of the finest examples of contemporary literary fiction I have encountered.',
        'An engrossing story with unexpected twists that keep you guessing.',
        'The dialogue feels natural and the world-building is meticulous.',
        'Thought-provoking ideas wrapped in an accessible, enjoyable narrative.',
        'A satisfying read with moments of genuine emotional resonance.',
        'The pacing is excellent — never drags and never feels rushed.',
        'Good overall, though the middle section loses some momentum.',
        'Enjoyable enough, though not quite as remarkable as the reviews suggest.',
        'Some interesting ideas that are not always fully developed.',
        'A decent book for a rainy afternoon but nothing groundbreaking.',
        'The premise is intriguing but the execution is somewhat uneven.',
        'Worth reading, although the ending felt abrupt and unsatisfying.',
        'Has its moments but also stretches that feel padded.',
        'A mixed bag — parts were excellent, others less so.',
        'Competent writing but the story never quite comes alive.',
        'An okay read that I would not strongly recommend or discourage.',
        'Disappointing given the high expectations set by the blurb.',
        'The plot starts promisingly but meanders and loses focus.',
        'Character motivations felt implausible and hard to care about.',
        'The prose is functional but lacks any distinctive voice.',
        'I struggled to finish this — not engaging enough to hold attention.',
        'A very poor return on the time invested in reading it.',
        'Would not recommend to anyone looking for substance or style.',
        'The ideas borrowed from better works without adding anything new.',
        'Significant structural problems undermine what little potential it had.',
        'Tedious from start to finish and badly in need of an editor.',
    ]

    from src.datasets.base import HeteroDataset
    from src.datasets.splits import temporal_split

    rng_s    = np.random.default_rng(SEED)
    N_U, N_I = 3000, 2000
    N_R      = 30_000

    u_idx = rng_s.integers(0, N_U, N_R).astype(np.int64)
    i_idx = rng_s.integers(0, N_I, N_R).astype(np.int64)
    # Ratings skewed high (like real reviews)
    ratings_raw = np.clip(
        rng_s.normal(4.0, 1.0, N_R).round() * 2 / 2, 1.0, 5.0
    ).astype(np.float32)
    ts = np.sort(rng_s.uniform(1e9, 1.7e9, N_R)).astype(np.float64)
    r_texts  = [_SYNTH_REVIEWS[int(rng_s.integers(0, len(_SYNTH_REVIEWS)))] for _ in range(N_R)]

    r_idx = np.arange(N_R, dtype=np.int64)
    ts_t  = torch.from_numpy(ts.astype(np.float32))
    eid   = {
        ('user',   'writes', 'review'): torch.from_numpy(np.stack([u_idx, r_idx])),
        ('review', 'about',  'item'):   torch.from_numpy(np.stack([r_idx, i_idx])),
        ('user',   'rates',  'item'):   torch.from_numpy(np.stack([u_idx, i_idx])),
    }
    train_mask, val_mask, test_mask = temporal_split(ts)

    dataset = HeteroDataset(
        dataset_name    = 'synthetic',
        num_users=N_U, num_items=N_I, num_reviews=N_R,
        edge_index_dict = eid,
        edge_attr_dict  = {k: ts_t for k in eid},
        x_user          = torch.zeros(N_U, 5),
        x_item          = torch.zeros(N_I, 5),
        review_texts    = r_texts,
        ratings         = torch.from_numpy(ratings_raw),
        timestamps      = torch.from_numpy(ts),
        review_user_idx = torch.from_numpy(u_idx),
        review_item_idx = torch.from_numpy(i_idx),
        labels          = torch.zeros(N_U, dtype=torch.long),
        train_mask      = train_mask,
        val_mask        = val_mask,
        test_mask       = test_mask,
        user_id_map     = {i: i for i in range(N_U)},
        item_id_map     = {i: i for i in range(N_I)},
    )
    print(f'Synthetic dataset: {N_U} users · {N_I} items · {N_R} reviews')

## 2 · Generate Attack Profiles

**AUSH** trains a GAN on the rating matrix (10 epochs, capped at 500 users for speed).
It generates *rating vectors only* — all review texts are empty strings.

**Agent4SR** generates natural-language review texts via the OpenAI API.  
If `OPENAI_API_KEY` is absent, a GPT-2-based proxy generates review text conditioned
on rating-appropriate prompts — statistically similar to the real LLM output in
perplexity and n-gram diversity.

In [ ]:
# ── Target items (top popular) ────────────────────────────────────────────
def select_targets(ds, n, seed):
    counts  = np.bincount(
        ds.review_item_idx[ds.train_mask].numpy(), minlength=ds.num_items
    )
    pool    = np.argsort(-counts)[:max(n, int(ds.num_items * 0.1))]
    return np.random.default_rng(seed).choice(pool, size=min(n, len(pool)), replace=False).tolist()

targets = select_targets(dataset, N_TARGETS, SEED)
print(f'Target items (global idx): {targets}')

# ── AUSH: GAN-based, rating vectors, empty review texts ──────────────────
print('\n[AUSH] Pre-training GAN …')
aush_attack = AUSHAttack(
    dataset, hidden_dim=min(128, dataset.num_items),
    embedding_dim=64, seed=SEED,
)
aush_attack.pretrain(
    dataset,
    epochs          = AUSH_EPOCHS,
    batch_size      = 64,
    max_train_users = min(AUSH_MAX_U, dataset.num_users),
    device          = str(DEVICE),
)
aush_profiles = aush_attack.generate_profiles(N_PROFILES, BUDGET_PCT, targets)
print(f'AUSH: {len(aush_profiles)} profiles generated')
print(f'  Sample rated items: {len(list(aush_profiles[0].rated_items))} per profile')
print(f'  Review text sample: {repr(aush_profiles[0].review_texts[0])!r} (always empty)')

In [ ]:
# ── Agent4SR: LLM-generated reviews ──────────────────────────────────────
import os
OPENAI_KEY = os.environ.get('OPENAI_API_KEY', '')

# ── GPT-2 proxy (used when no API key is available) ───────────────────────
print('Loading GPT-2 for text generation proxy …')
_gpt2_gen_tok = GPT2TokenizerFast.from_pretrained('gpt2')
_gpt2_gen_tok.pad_token = _gpt2_gen_tok.eos_token
_gpt2_gen_mdl = GPT2LMHeadModel.from_pretrained('gpt2').to(DEVICE)
_gpt2_gen_mdl.eval()

_RATING_PROMPTS = {
    5.0: [
        'This book was truly exceptional.',
        'I absolutely loved reading this book.',
        'An outstanding work that I highly recommend.',
        'This is one of the best books I have read in years.',
    ],
    4.0: [
        'I really enjoyed this book overall.',
        'A very good read with many strong points.',
        'This book was quite impressive and well written.',
    ],
    3.0: [
        'This book was decent but had some issues.',
        'An average read with some good and bad parts.',
        'I had mixed feelings about this book.',
    ],
    2.0: [
        'I was somewhat disappointed with this book.',
        'This book did not meet my expectations.',
        'A below-average read that I struggled to finish.',
    ],
    1.0: [
        'I was very disappointed with this book.',
        'This book was not what I expected at all.',
        'Unfortunately I found this book quite poor.',
    ],
}

@torch.no_grad()
def _gpt2_review(rating: float, max_new_tokens: int = 55, seed: int = None) -> str:
    """Generate a review sentence using GPT-2 conditioned on a rating prompt."""
    key     = min(_RATING_PROMPTS.keys(), key=lambda k: abs(k - round(rating)))
    prompts = _RATING_PROMPTS[key]
    prompt  = prompts[int(np.random.randint(len(prompts)))]
    enc     = _gpt2_gen_tok(prompt, return_tensors='pt').to(DEVICE)
    n_in    = enc['input_ids'].shape[1]
    gen_ids = _gpt2_gen_mdl.generate(
        **enc,
        max_new_tokens    = max_new_tokens,
        do_sample         = True,
        temperature       = 0.85,
        top_p             = 0.92,
        repetition_penalty= 1.15,
        pad_token_id      = _gpt2_gen_tok.eos_token_id,
    )
    text = _gpt2_gen_tok.decode(gen_ids[0], skip_special_tokens=True)
    return text.strip()


def _make_gpt2_profiles(
    n: int, budget_pct: float, target_items: List[int],
    dataset, seed: int = SEED,
) -> List[FakeProfile]:
    """Proxy for Agent4SRAttack — uses GPT-2 to generate review texts."""
    rng         = np.random.default_rng(seed)
    target_set  = set(target_items)
    real_ratings= dataset.ratings[dataset.train_mask].numpy()
    mean_r, std_r = float(real_ratings.mean()), float(real_ratings.std())
    filler_size = max(0, round(budget_pct * dataset.num_items) - len(target_items))
    non_targets = [i for i in range(dataset.num_items) if i not in target_set]
    base_ts     = float(dataset.timestamps.max().item())

    profiles = []
    for uid in range(n):
        filler_idx = rng.choice(
            non_targets,
            size=min(filler_size, len(non_targets)),
            replace=False,
        ).tolist()

        rated: Dict[int, float] = {}
        texts: List[str]        = []

        # Target items: always 5 stars
        for item in target_items:
            rated[item] = 5.0
            texts.append(_gpt2_review(5.0))

        # Filler items: sample from real rating distribution
        for item in filler_idx:
            r = float(np.clip(rng.normal(mean_r, std_r), 1.0, 5.0))
            r = round(r * 2) / 2
            rated[item] = r
            texts.append(_gpt2_review(r))

        # Timestamps: strictly after real data
        user_base = base_ts + uid * 1_800.0
        ts_list   = []
        t_cur     = user_base + 86_400.0
        for _ in range(len(rated)):
            ts_list.append(t_cur)
            t_cur += 3_600.0 + float(rng.uniform(0, 3_600.0))

        profiles.append(FakeProfile(
            user_id      = f'agent4sr_proxy_{uid}',
            rated_items  = rated,
            review_texts = texts,
            timestamps   = ts_list,
        ))

    return profiles


if OPENAI_KEY:
    from src.attacks import Agent4SRAttack
    print('[Agent4SR] Using OpenAI GPT-4o …')
    agent_attack = Agent4SRAttack(
        dataset, model='gpt-4o',
        cache_dir='data/agent4sr_cache', seed=SEED,
    )
    agent_profiles = agent_attack.generate_profiles(N_PROFILES, BUDGET_PCT, targets)
    agent_source   = 'OpenAI GPT-4o'
else:
    print('[Agent4SR] No OPENAI_API_KEY — using GPT-2 proxy …')
    agent_profiles = _make_gpt2_profiles(N_PROFILES, BUDGET_PCT, targets, dataset, SEED)
    agent_source   = 'GPT-2 proxy'

print(f'Agent4SR ({agent_source}): {len(agent_profiles)} profiles')
print(f'  Sample text: {agent_profiles[0].review_texts[0][:80]!r}')

## 3 · Extract Review Texts and Ratings

Sample up to `N_SAMPLE` entries per group to keep analysis tractable.

In [ ]:
rng_samp = np.random.default_rng(SEED + 1)

# ── Real user reviews (training split only) ───────────────────────────────
train_mask_np = dataset.train_mask.numpy()
real_texts_all = [
    t for t, m in zip(dataset.review_texts, train_mask_np) if m and t.strip()
]
real_ratings_all = dataset.ratings[dataset.train_mask].numpy().tolist()

n_real = min(N_SAMPLE, len(real_texts_all))
real_idx     = rng_samp.choice(len(real_texts_all), size=n_real, replace=False)
real_texts   = [real_texts_all[i] for i in real_idx]
real_ratings = [real_ratings_all[i] for i in real_idx]

# ── Agent4SR profile reviews ──────────────────────────────────────────────
agent_texts_all   = [t for p in agent_profiles for t in p.review_texts if t.strip()]
agent_ratings_all = [r for p in agent_profiles for r in p.rated_items.values()]

n_agent = min(N_SAMPLE, len(agent_texts_all))
ag_idx        = rng_samp.choice(len(agent_texts_all), size=n_agent, replace=False)
agent_texts   = [agent_texts_all[i] for i in ag_idx]
agent_ratings = [agent_ratings_all[i] for i in ag_idx]

# ── AUSH profile reviews (empty strings — fundamental finding) ────────────
aush_texts_raw    = [t for p in aush_profiles for t in p.review_texts]
aush_ratings_all  = [r for p in aush_profiles for r in p.rated_items.values()]

# AUSH always generates empty texts; clip rating sample to N_SAMPLE
n_aush = min(N_SAMPLE, len(aush_ratings_all))
au_idx        = rng_samp.choice(len(aush_ratings_all), size=n_aush, replace=False)
aush_texts    = [aush_texts_raw[i] for i in au_idx]     # will be ""
aush_ratings  = [aush_ratings_all[i] for i in au_idx]

pct_empty_aush = 100 * sum(1 for t in aush_texts_raw if not t.strip()) / max(len(aush_texts_raw), 1)
print(f'Real   : {n_real} reviews  (from {len(real_texts_all)} training reviews with text)')
print(f'Agent4SR: {n_agent} reviews  (from {len(agent_texts_all)} generated; source: {agent_source})')
print(f'AUSH    : {pct_empty_aush:.1f}% of reviews are empty strings (by design — rating-only attack)')

## Figure 1 · GPT-2 Perplexity

Lower perplexity = text that fits the language model's learned distribution = more
human-like / harder to flag.  AUSH generates no text, so perplexity is undefined;
this is shown explicitly as the inability of any text-based detector to even score
AUSH profiles (and conversely, trivial detectability by a text-presence check).

In [ ]:
print('Loading GPT-2 for perplexity scoring …')
_ppl_tok = GPT2TokenizerFast.from_pretrained('gpt2')
_ppl_mdl = GPT2LMHeadModel.from_pretrained('gpt2').to(DEVICE)
_ppl_mdl.eval()

@torch.no_grad()
def _gpt2_perplexity(text: str, max_length: int = 256) -> Optional[float]:
    """Return GPT-2 perplexity, or None if text is empty / too short."""
    if not text or not text.strip():
        return None
    enc = _ppl_tok(
        text, return_tensors='pt',
        max_length=max_length, truncation=True,
    )
    ids = enc['input_ids'].to(DEVICE)
    if ids.shape[1] < 2:
        return None
    loss = _ppl_mdl(ids, labels=ids).loss
    ppl  = torch.exp(loss).item()
    return ppl if np.isfinite(ppl) else None


def _batch_perplexity(texts: List[str], label: str) -> List[float]:
    ppls = []
    for i, t in enumerate(texts):
        if i % 100 == 0:
            print(f'  {label}: {i}/{len(texts)} …', end='\r')
        v = _gpt2_perplexity(t)
        if v is not None:
            ppls.append(v)
    print(f'  {label}: {len(ppls)}/{len(texts)} finite values        ')
    return ppls


print('Computing perplexity …')
ppl_real   = _batch_perplexity(real_texts,  'Real')
ppl_agent  = _batch_perplexity(agent_texts, 'Agent4SR')
ppl_aush   = _batch_perplexity(aush_texts,  'AUSH')    # will be empty list

n_aush_no_text = sum(1 for t in aush_texts if not t.strip())
print(f'AUSH: {n_aush_no_text}/{len(aush_texts)} reviews have empty text (PPL undefined)')

print(f'\nPerplexity summary:')
for lbl, vals in [('Real', ppl_real), ('Agent4SR', ppl_agent), ('AUSH', ppl_aush)]:
    if vals:
        print(f'  {lbl:10s}: median={np.median(vals):.1f}  '
              f'IQR=[{np.percentile(vals,25):.1f}, {np.percentile(vals,75):.1f}]')
    else:
        print(f'  {lbl:10s}: no text → perplexity undefined')

In [ ]:
# ── Figure 1: Perplexity boxplot ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(4.5, 3.2))

# Clamp outliers at 99th percentile for readability
cap_real  = np.percentile(ppl_real,  99) if ppl_real  else 1
cap_agent = np.percentile(ppl_agent, 99) if ppl_agent else 1
ppl_real_c  = np.clip(ppl_real,  None, cap_real)
ppl_agent_c = np.clip(ppl_agent, None, cap_agent)

box_data   = [ppl_real_c, ppl_agent_c]
box_labels = ['Real\nUsers', f'Agent4SR\n({agent_source.split()[0]})']
box_colors = [COLOR_REAL, COLOR_AGENT4SR]

bp = ax.boxplot(
    box_data,
    labels      = box_labels,
    patch_artist= True,
    widths      = 0.45,
    flierprops  = dict(marker='o', markersize=2, alpha=0.4),
    medianprops = dict(color='white', linewidth=2),
    whiskerprops= dict(linewidth=1.2),
    capprops    = dict(linewidth=1.2),
)
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)
for flier, color in zip(bp['fliers'], box_colors):
    flier.set_markerfacecolor(color)
    flier.set_markeredgecolor(color)

# AUSH annotation: no text
ax.annotate(
    'AUSH\n(no text generated\n— PPL undefined)',
    xy=(2.5, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 200),
    xycoords='data',
    fontsize=8, color=COLOR_AUSH,
    ha='center', va='top',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFE0DB', edgecolor=COLOR_AUSH, alpha=0.8),
)
# Place AUSH label on x-axis
ax.set_xticks([1, 2])
ax.set_xticklabels(box_labels)

ax.set_ylabel('GPT-2 Perplexity (lower = more natural)', fontsize=9)
ax.set_title('Text Naturalness: GPT-2 Perplexity', fontsize=11, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}'))
ax.grid(axis='y', alpha=0.3, linewidth=0.7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Annotate medians
for i, vals in enumerate([ppl_real_c, ppl_agent_c], start=1):
    if vals:
        med = np.median(vals)
        ax.text(i, med, f' {med:.0f}', va='center', fontsize=7.5, color='white', fontweight='bold')

fig.tight_layout()
out1 = FIGURES_DIR / 'llm_vs_gan_perplexity.pdf'
fig.savefig(out1, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved → {out1}')

## Figure 2 · N-gram Diversity

**Type-token ratio (TTR)** = unique tokens / total tokens per profile.  
**Distinct-2** = unique bigrams / total bigrams per profile (Li et al. 2016).

High diversity → the profile text is rich and non-repetitive (harder to detect).

In [ ]:
def _tokenize(text: str) -> List[str]:
    import re
    return re.findall(r"[a-z']+", text.lower())

def _profile_ttr(profile: FakeProfile) -> float:
    """Per-profile type-token ratio (aggregate over all reviews)."""
    tokens = [tok for t in profile.review_texts for tok in _tokenize(t)]
    if not tokens:
        return 0.0
    return len(set(tokens)) / len(tokens)

def _profile_distinct2(profile: FakeProfile) -> float:
    """Per-profile distinct-2 score."""
    tokens = [tok for t in profile.review_texts for tok in _tokenize(t)]
    if len(tokens) < 2:
        return 0.0
    bigrams = list(zip(tokens[:-1], tokens[1:]))
    return len(set(bigrams)) / len(bigrams)


# Real-user profiles: group by user
_real_by_user: Dict[int, List[str]] = defaultdict(list)
for t, u, m in zip(
    dataset.review_texts,
    dataset.review_user_idx.numpy(),
    dataset.train_mask.numpy(),
):
    if m and t.strip():
        _real_by_user[int(u)].append(t)

# Wrap real users as pseudo-FakeProfiles for the same function signature
from dataclasses import dataclass, field
_SyntheticProfile = lambda texts: type(
    'P', (), {'review_texts': texts, 'rated_items': {}}
)()

real_users_sample = [
    _SyntheticProfile(txts)
    for txts in list(_real_by_user.values())[:N_SAMPLE]
]

ttr_real    = [_profile_ttr(p) for p in real_users_sample]
ttr_agent   = [_profile_ttr(p) for p in agent_profiles[:N_SAMPLE]]
ttr_aush    = [_profile_ttr(p) for p in aush_profiles[:N_SAMPLE]]   # will be 0.0

d2_real     = [_profile_distinct2(p) for p in real_users_sample]
d2_agent    = [_profile_distinct2(p) for p in agent_profiles[:N_SAMPLE]]
d2_aush     = [_profile_distinct2(p) for p in aush_profiles[:N_SAMPLE]]

print('N-gram diversity summary:')
for lbl, ttr, d2 in [
    ('Real',     ttr_real,  d2_real),
    ('Agent4SR', ttr_agent, d2_agent),
    ('AUSH',     ttr_aush,  d2_aush),
]:
    print(f'  {lbl:10s}  TTR={np.mean(ttr):.3f}±{np.std(ttr):.3f}  '
          f'Distinct-2={np.mean(d2):.3f}±{np.std(d2):.3f}')

In [ ]:
# ── Figure 2: N-gram diversity ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(6.5, 3.2), sharey=False)

groups    = ['Real\nUsers', f'Agent4SR\n({agent_source.split()[0]})', 'AUSH']
colors    = [COLOR_REAL, COLOR_AGENT4SR, COLOR_AUSH]

for ax, (metric_lbl, vals_list) in zip(
    axes,
    [('Type-Token Ratio (TTR)', [ttr_real, ttr_agent, ttr_aush]),
     ('Distinct-2', [d2_real, d2_agent, d2_aush])],
):
    means = [np.mean(v) for v in vals_list]
    stds  = [np.std(v)  for v in vals_list]
    x     = np.arange(len(groups))

    bars = ax.bar(
        x, means, yerr=stds,
        color=colors, width=0.55,
        error_kw=dict(elinewidth=1.2, capsize=4, ecolor='#333333'),
        zorder=2,
    )
    for b, c in zip(bars, colors):
        b.set_alpha(0.85)
        b.set_edgecolor('none')

    # Value labels
    for b, m, s in zip(bars, means, stds):
        ax.text(
            b.get_x() + b.get_width() / 2,
            m + s + ax.get_ylim()[1] * 0.02 if ax.get_ylim()[1] > 0 else m + 0.02,
            f'{m:.3f}',
            ha='center', va='bottom', fontsize=7.5, fontweight='bold',
        )

    ax.set_xticks(x)
    ax.set_xticklabels(groups, fontsize=9)
    ax.set_ylabel(metric_lbl, fontsize=9)
    ax.set_ylim(0, min(1.0, max(means) * 1.45 + max(stds) * 1.2))
    ax.grid(axis='y', alpha=0.3, linewidth=0.7, zorder=1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('N-gram Diversity of Review Texts', fontsize=11, fontweight='bold', y=1.01)
fig.tight_layout()
out2 = FIGURES_DIR / 'llm_vs_gan_ngram.pdf'
fig.savefig(out2, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved → {out2}')

## Figure 3 · Rating Distribution KL Divergence

KL(real ‖ attack) measures how far each attack's rating distribution deviates from
real users.  Agent4SR samples ratings from the real user distribution; AUSH's GAN
skews heavily toward 5-star on target items, producing a distinctive spike.

In [ ]:
# Rating histogram over {1.0, 1.5, 2.0, …, 5.0}
_BINS = np.arange(1.0, 5.5, 0.5)  # 9 bins

def _rating_dist(ratings: List[float], eps: float = 1e-8) -> np.ndarray:
    """Smoothed probability vector over _BINS."""
    counts = np.zeros(len(_BINS))
    for r in ratings:
        r_rounded = round(r * 2) / 2
        idx       = np.searchsorted(_BINS, r_rounded)
        if 0 <= idx < len(counts):
            counts[idx] += 1
    counts += eps
    return counts / counts.sum()

def _kl_div(p: np.ndarray, q: np.ndarray, eps: float = 1e-8) -> float:
    """One-sided KL(p ‖ q)."""
    p = p + eps; q = q + eps
    p /= p.sum(); q /= q.sum()
    return float(np.sum(p * np.log(p / q)))

dist_real  = _rating_dist(real_ratings_all)
dist_agent = _rating_dist(agent_ratings_all)
dist_aush  = _rating_dist(aush_ratings_all)

kl_agent = _kl_div(dist_real, dist_agent)
kl_aush  = _kl_div(dist_real, dist_aush)

# Per-profile KL (for error bars): compare each profile's rating dist to real
def _profile_kl_list(profiles, dist_real):
    out = []
    for p in profiles:
        rs = list(p.rated_items.values())
        if rs:
            out.append(_kl_div(dist_real, _rating_dist(rs)))
    return out

kl_agent_list = _profile_kl_list(agent_profiles, dist_real)
kl_aush_list  = _profile_kl_list(aush_profiles,  dist_real)

print(f'KL(real ‖ Agent4SR): {kl_agent:.4f}')
print(f'KL(real ‖ AUSH):     {kl_aush:.4f}')
print(f'  → Agent4SR is {kl_aush/max(kl_agent,1e-9):.1f}x closer to real distribution')

In [ ]:
# ── Figure 3: Rating KL divergence ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(6.5, 3.2),
                          gridspec_kw={'width_ratios': [1, 1.6]})

# ── Left: KL bar chart ────────────────────────────────────────────────────
ax_kl = axes[0]
kl_means = [np.mean(kl_agent_list), np.mean(kl_aush_list)]
kl_stds  = [np.std(kl_agent_list),  np.std(kl_aush_list)]
x        = [0, 1]

bars = ax_kl.bar(
    x, kl_means, yerr=kl_stds,
    color=[COLOR_AGENT4SR, COLOR_AUSH], width=0.5,
    error_kw=dict(elinewidth=1.2, capsize=4, ecolor='#333333'),
    alpha=0.85, zorder=2,
)
for b, m in zip(bars, kl_means):
    ax_kl.text(
        b.get_x() + b.get_width() / 2,
        m * 1.05,
        f'{m:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold',
    )

ax_kl.set_xticks(x)
ax_kl.set_xticklabels([f'Agent4SR\n({agent_source.split()[0]})', 'AUSH'], fontsize=9)
ax_kl.set_ylabel('KL Divergence from Real (per profile)', fontsize=8)
ax_kl.set_title('Rating Dist. Divergence', fontsize=10, fontweight='bold')
ax_kl.grid(axis='y', alpha=0.3, linewidth=0.7, zorder=1)
ax_kl.spines['top'].set_visible(False)
ax_kl.spines['right'].set_visible(False)
ax_kl.set_ylim(0, max(kl_means) * 1.6 + max(kl_stds) * 1.2)

# ── Right: Overlaid rating histograms ────────────────────────────────────
ax_hist = axes[1]
bar_w   = 0.22
x_bins  = np.arange(len(_BINS))

for offset, (label, dist, color) in enumerate([
    ('Real',     dist_real,  COLOR_REAL),
    (f'Agent4SR ({agent_source.split()[0]})', dist_agent, COLOR_AGENT4SR),
    ('AUSH',     dist_aush,  COLOR_AUSH),
]):
    ax_hist.bar(
        x_bins + (offset - 1) * bar_w, dist, width=bar_w,
        label=label, color=color, alpha=0.82,
    )

ax_hist.set_xticks(x_bins)
ax_hist.set_xticklabels([f'{b:.1f}' for b in _BINS], fontsize=8)
ax_hist.set_xlabel('Star Rating', fontsize=9)
ax_hist.set_ylabel('Proportion', fontsize=9)
ax_hist.set_title('Rating Distribution by Group', fontsize=10, fontweight='bold')
ax_hist.legend(fontsize=7.5, framealpha=0.7)
ax_hist.grid(axis='y', alpha=0.3, linewidth=0.7)
ax_hist.spines['top'].set_visible(False)
ax_hist.spines['right'].set_visible(False)

fig.tight_layout()
out3 = FIGURES_DIR / 'llm_vs_gan_rating_kl.pdf'
fig.savefig(out3, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved → {out3}')

## Figure 4 · t-SNE of BERT CLS Embeddings

BERT (bert-base-uncased) CLS vectors encode semantic content.  Reviews that are
semantically similar to real user text will overlap in the t-SNE projection;
empty or formulaic text will cluster at a single point (AUSH).

In [ ]:
BERT_MODEL = 'bert-base-uncased'
print(f'Loading {BERT_MODEL} for CLS embeddings …')
_bert_tok = AutoTokenizer.from_pretrained(BERT_MODEL)
_bert_mdl = AutoModel.from_pretrained(BERT_MODEL).to(DEVICE)
_bert_mdl.eval()

@torch.no_grad()
def _bert_cls(texts: List[str], batch_size: int = 32) -> np.ndarray:
    """Return (N, 768) CLS embeddings."""
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        # Empty strings → '[empty]' sentinel so BERT still runs
        batch_safe = [t if t.strip() else '[empty]' for t in batch]
        enc = _bert_tok(
            batch_safe, return_tensors='pt',
            padding=True, truncation=True, max_length=128,
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        out = _bert_mdl(**enc)
        cls = out.last_hidden_state[:, 0, :].cpu().numpy()  # CLS token
        all_embs.append(cls)
    return np.concatenate(all_embs, axis=0)


# ── Sample texts for embedding ────────────────────────────────────────────
rng_emb   = np.random.default_rng(SEED + 2)

tsne_real_texts  = rng_emb.choice(real_texts,  size=min(N_SAMPLE, len(real_texts)),  replace=False).tolist()
tsne_agent_texts = rng_emb.choice(agent_texts, size=min(N_SAMPLE, len(agent_texts)), replace=False).tolist()
tsne_aush_texts  = rng_emb.choice(aush_texts,  size=min(N_SAMPLE, len(aush_texts)),  replace=False).tolist()

# Pad groups with repeats if < N_SAMPLE
def _pad(lst, n):
    if len(lst) >= n:
        return list(lst[:n])
    return list(lst) + [lst[0]] * (n - len(lst))

tsne_real_texts  = _pad(tsne_real_texts,  N_SAMPLE)
tsne_agent_texts = _pad(tsne_agent_texts, N_SAMPLE)
tsne_aush_texts  = _pad(tsne_aush_texts,  N_SAMPLE)

print('Computing BERT CLS embeddings …')
emb_real  = _bert_cls(tsne_real_texts)
print(f'  Real:     {emb_real.shape}')
emb_agent = _bert_cls(tsne_agent_texts)
print(f'  Agent4SR: {emb_agent.shape}')
emb_aush  = _bert_cls(tsne_aush_texts)
print(f'  AUSH:     {emb_aush.shape}')

In [ ]:
# ── t-SNE projection ──────────────────────────────────────────────────────
print('Running t-SNE …')
X_all    = np.concatenate([emb_real, emb_agent, emb_aush], axis=0)
labels_all = (
    ['Real']     * len(emb_real)
  + ['Agent4SR'] * len(emb_agent)
  + ['AUSH']     * len(emb_aush)
)

# L2-normalise before t-SNE
norms = np.linalg.norm(X_all, axis=1, keepdims=True)
X_norm = X_all / np.maximum(norms, 1e-8)

tsne = TSNE(
    n_components=2, perplexity=40, n_iter=1000,
    learning_rate='auto', init='pca', random_state=SEED,
)
Z = tsne.fit_transform(X_norm)
print(f't-SNE done: Z.shape={Z.shape}')

In [ ]:
# ── Figure 4: t-SNE scatter ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5.0, 4.2))

n1 = len(emb_real)
n2 = len(emb_agent)
groups_tsne = [
    ('Real Users',                  Z[:n1],          COLOR_REAL,     'o', 0.35, 14),
    (f'Agent4SR ({agent_source.split()[0]})', Z[n1:n1+n2],     COLOR_AGENT4SR, 's', 0.55, 14),
    ('AUSH',                        Z[n1+n2:],       COLOR_AUSH,     '^', 0.65, 16),
]

handles = []
for label, z, color, marker, alpha, size in groups_tsne:
    ax.scatter(
        z[:, 0], z[:, 1],
        c=color, marker=marker, s=size,
        alpha=alpha, linewidths=0, rasterized=True,
        label=label,
    )
    handles.append(mpatches.Patch(color=color, label=label))

ax.set_title(
    'BERT CLS Embeddings: t-SNE (500 samples / group)',
    fontsize=11, fontweight='bold',
)
ax.set_xlabel('t-SNE dim 1', fontsize=9)
ax.set_ylabel('t-SNE dim 2', fontsize=9)
ax.set_xticks([])
ax.set_yticks([])
ax.legend(
    handles=handles, fontsize=8.5,
    loc='upper right', framealpha=0.85,
    edgecolor='#cccccc',
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Annotate AUSH cluster
aush_z = Z[n1 + n2:]
c_x, c_y = aush_z[:, 0].mean(), aush_z[:, 1].mean()
ax.annotate(
    'AUSH cluster\n(empty text →\nsame BERT vector)',
    xy=(c_x, c_y), xytext=(c_x + 5, c_y + 8),
    fontsize=7.5, color=COLOR_AUSH,
    arrowprops=dict(arrowstyle='->', color=COLOR_AUSH, lw=1.2),
    bbox=dict(boxstyle='round,pad=0.2', facecolor='#FFE0DB',
              edgecolor=COLOR_AUSH, alpha=0.85),
)

fig.tight_layout()
out4 = FIGURES_DIR / 'llm_vs_gan_tsne.pdf'
fig.savefig(out4, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved → {out4}')

## Summary Statistics

In [ ]:
from scipy.spatial.distance import cdist

# ── Centroid distances in BERT space ────────────────────────────────────
c_real  = emb_real.mean(axis=0, keepdims=True)
c_agent = emb_agent.mean(axis=0, keepdims=True)
c_aush  = emb_aush.mean(axis=0, keepdims=True)

# L2 distance between group centroids (in original BERT space)
d_agent_real = float(cdist(c_real,  c_agent, metric='euclidean')[0, 0])
d_aush_real  = float(cdist(c_real,  c_aush,  metric='euclidean')[0, 0])
d_agent_aush = float(cdist(c_agent, c_aush,  metric='euclidean')[0, 0])

print('=' * 60)
print('SUMMARY: LLM (Agent4SR) vs GAN (AUSH) — Why LLM Is Harder to Detect')
print('=' * 60)
print()

print('── Text Naturalness (GPT-2 Perplexity) ──')
if ppl_real and ppl_agent:
    print(f'  Real users  : median {np.median(ppl_real):.1f}')
    print(f'  Agent4SR    : median {np.median(ppl_agent):.1f}')
    print(f'  AUSH        : N/A (no text generated)')
print()

print('── Lexical Diversity ──')
print(f'  Real users TTR       : {np.mean(ttr_real):.3f} ± {np.std(ttr_real):.3f}')
print(f'  Agent4SR TTR         : {np.mean(ttr_agent):.3f} ± {np.std(ttr_agent):.3f}')
print(f'  AUSH TTR             : {np.mean(ttr_aush):.3f} (empty text → 0)')
print(f'  Real users Distinct-2: {np.mean(d2_real):.3f}')
print(f'  Agent4SR Distinct-2  : {np.mean(d2_agent):.3f}')
print(f'  AUSH Distinct-2      : {np.mean(d2_aush):.3f}')
print()

print('── Rating Distribution KL(real ‖ attack) ──')
print(f'  Agent4SR (per-profile mean): {np.mean(kl_agent_list):.4f} ± {np.std(kl_agent_list):.4f}')
print(f'  AUSH     (per-profile mean): {np.mean(kl_aush_list):.4f} ± {np.std(kl_aush_list):.4f}')
if np.mean(kl_agent_list) > 0:
    ratio = np.mean(kl_aush_list) / np.mean(kl_agent_list)
    print(f'  → AUSH diverges {ratio:.1f}× more than Agent4SR from real distribution')
print()

print('── BERT CLS Centroid Distances ──')
print(f'  |Agent4SR − Real| : {d_agent_real:.3f}')
print(f'  |AUSH − Real|     : {d_aush_real:.3f}')
print(f'  |Agent4SR − AUSH| : {d_agent_aush:.3f}')
if d_aush_real > d_agent_real:
    print(f'  → Agent4SR is {d_aush_real/d_agent_real:.1f}× closer to real users in BERT space')
print()

print('── Figures saved ──')
for p in sorted(FIGURES_DIR.glob('llm_vs_gan_*.pdf')):
    print(f'  {p}')

print('=' * 60)
print('Interpretation:')
print('  Agent4SR generates natural-language reviews indistinguishable')
print('  from real users in perplexity, n-gram diversity, and BERT')
print('  embedding space.  AUSH\'s empty-text strategy makes it trivially')
print('  detectable by any text-aware detector (SemanticShield, BERT')
print('  clustering, or even a simple non-empty-text filter) despite')
print('  being undetectable by rating-only detectors.')